## **LangChain 🦜🔗: Chains**

In [ ]:
!pip install openai
!pip install langchain-core
!pip install langchain-openai

In [ ]:
import openai
import os

In [ ]:
f = open('/content/ts_openapi_key.txt')
api_key = f.read()
os.environ['OPENAI_API_KEY'] = api_key
openai.api_key= os.getenv('OPENAI_API_KEY')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate     #
from langchain_core.prompts import ChatPromptTemplate  #
from langchain_core.output_parsers import StrOutputParser

### **Most Basic chain**
[LCEL:Getting Started](https://python.langchain.com/v0.1/docs/expression_language/get_started/)

**Example:1**

Notice: `.from_template`

In [ ]:
llm_model = "gpt-4o-mini" # This is a chat model
llm = ChatOpenAI(temperature=0.9, model=llm_model)

In [ ]:
prompt1 = ChatPromptTemplate.from_template(
    "What is the best name to describe \
    a company that makes {product}?"
)

In [ ]:
prompt1

ChatPromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['product'], input_types={}, partial_variables={}, template='What is the best name to describe     a company that makes {product}?'), additional_kwargs={})])

In [ ]:
chain1 = prompt1 | llm |StrOutputParser() # Try with or without-> |StrOutputParser()

**LCEL(LangChain Expression Language)**--> uses pipes(|):  When we call invoke() on the chain, the input is first passed to prompt, output of that is passed to the llm. It's just the combination of the LLM
and the prompt. But now this chain will let us run through the
prompt and into the LLM in a sequential manner.

In [ ]:
product = "cup"
result= chain1.invoke(product)
print(result)

Choosing a name for a company that makes cups can depend on various factors such as the target audience, brand personality, and product uniqueness. Here are some suggestions that might resonate well:

1. **CupCraft**
2. **Mug & More**
3. **Cuppa Creations**
4. **The Cup Co.**
5. **Sip & Style**
6. **Chic Cups**
7. **Timeless Tumblers**
8. **Cupsational**
9. **Cupology**
10. **EverCup**

You might also consider names that highlight the materials used (e.g., ceramic, eco-friendly), the design aesthetic (e.g., modern, whimsical), or the purpose (e.g., travel, home use). Make sure to check for domain availability and trademark registration if you plan to create a website or brand around it!


The **StrOutputParser** in LangChain is a tool used to convert the output from language models (LLMs) into a simplified string format. This parser is particularly helpful when the model's output is structured in complex formats, such as JSON, and you need a simple text-based result for display or further processing. It is commonly used when working with LLMs that output messages or structured data, making it easier to handle by extracting only the relevant content.

**Example:2**

Notice: `.from_messages`

In [ ]:
prompt2=ChatPromptTemplate.from_messages([("system","You are a knowladgeable historian" ),
                                          ("human","say 3 lines about {input}")])
prompt2

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a knowladgeable historian'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='say 3 lines about {input}'), additional_kwargs={})])

In [ ]:
chain2 = prompt2| llm |StrOutputParser() # LCEL --> LangChain Expression Language

In [ ]:
event = "Taj Mahal"
result= chain2.invoke({'input':event})
result

'The Taj Mahal, located in Agra, India, is a UNESCO World Heritage Site and an iconic symbol of Mughal architecture. Built between 1632 and 1648, it was commissioned by Emperor Shah Jahan in memory of his beloved wife Mumtaz Mahal. Its stunning white marble façade, intricate inlay work, and surrounding gardens make it one of the most visited landmarks in the world.'

### **Runnable**

A Runnable is a unit of execution in the LangChain framework. It represents a specific task or operation that can be performed.
Examples of Runnables include data transformations, computations, or any other operation that can be expressed in the LangChain expression language.

**Example 1: [Runnable Lambdas](https://api.python.langchain.com/en/latest/runnables/langchain_core.runnables.base.RunnableLambda.html)**

The RunnableLambda is a LangChain abstraction that allows us to turn Python functions into pipe-compatible functions, similar to the Runnable class.

In [ ]:
from langchain_core.runnables import RunnableLambda

In [ ]:
def add_five(x):
    return x + 5

def multiply_by_two(x):
    return x * 2

# wrap the functions with RunnableLambda
chain = RunnableLambda(add_five) | RunnableLambda(multiply_by_two)


In [ ]:
chain.invoke(2)

14

In [ ]:
# A RunnableSequence constructed using the `|` operator
sequence = RunnableLambda(lambda x: x + 1) | RunnableLambda(lambda x: x * 2)
sequence.invoke(1)

4

In [ ]:
exmp=lambda x,y : x+y
exmp(2,3)

5

**Example 2 : [RunnablePassthrough](https://api.python.langchain.com/en/latest/runnables/langchain_core.runnables.passthrough.RunnablePassthrough.html)** on its own allows you to pass inputs unchanged. This typically is **used in conjuction with [RunnableParallel](https://python.langchain.com/v0.1/docs/expression_language/interface/#parallelism)** to pass data through to a new key in the map.

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

runnable = RunnableParallel(
    passed=RunnablePassthrough(),
    modified = lambda x:   x["num"] *4  ,
)

In [ ]:
runnable.invoke({"num": 2})

{'passed': {'num': 2}, 'modified': 8}

**Example 3: RunnableMap**: Parallel Execution -
The RunnableMap class runs a mapping of Runnables in parallel. It takes a list of inputs and applies each Runnable to its corresponding input.
The result is a mapping of the outputs from each Runnable.
Essentially, it allows you to process multiple inputs concurrently and collect their outputs in a structured way.

In [ ]:
## Fictional character
context1 =  """ Dr. Ananya Verma, a legendary Indian physicist born
in Kolkata in 1960, has made significant contributions to theoretical physics.
 Her achievements include the discovery of the Bose–Verma condensate, formulation
  of quantum entanglement theorems, and resolution of the Verma–Hawking black hole paradox.
  Dr. Verma’s brilliance has earned her praise from fellow scientists,
 and her legacy continues to inspire generations of physicists worldwide. """

question1 ="What are few discoveris by Dr. Ananya Verma?"

In [ ]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [ ]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [ ]:
from langchain_core.runnables import RunnableMap

In [ ]:
chain = RunnableMap(  {"context": lambda x: x["context"], "question": lambda x: x["question"]}   )   | prompt | llm | StrOutputParser()
chain

{
  context: RunnableLambda(...),
  question: RunnableLambda(...)
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x7bdb95f69d50>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x7bdb95fb0fd0>, root_client=<openai.OpenAI object at 0x7bdb95f68d60>, root_async_client=<openai.AsyncOpenAI object at 0x7bdb95f68c70>, model_name='gpt-4o-mini', temperature=0.9, model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser()

In [ ]:
chain = {"context": lambda x: x["context"], "question": lambda x: x["question"]} | prompt | llm | StrOutputParser()
chain

{
  context: RunnableLambda(...),
  question: RunnableLambda(...)
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])
| ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x7e298a2d6740>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x7e298a0f0850>, root_client=<openai.OpenAI object at 0x7e298a26edd0>, root_async_client=<openai.AsyncOpenAI object at 0x7e298a2d67a0>, model_name='gpt-4o-mini', temperature=0.9, model_kwargs={}, openai_api_key=SecretStr('**********'))
| StrOutputParser()

In [ ]:
chain.invoke(  {"context":context1,"question": question1}  )

"Dr. Ananya Verma's discoveries include the Bose–Verma condensate, the formulation of quantum entanglement theorems, and the resolution of the Verma–Hawking black hole paradox."

### **Complex - Sequential Chain**
**Example :1**

In [ ]:
llm_model = "gpt-4o-mini" # This is a chat model
llm = ChatOpenAI(temperature=0.9, model=llm_model)

In [ ]:
review = """Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...
Vieux lot ou contrefaçon !?"""

In [ ]:
# prompt template 1: translate to english
first_prompt = ChatPromptTemplate.from_template(
    "Translate the following review to english:"
    "\n\n{Review}"
)
# chain 1: input= Review and output= English_Review
chain_one = first_prompt|llm|StrOutputParser()   # output_key="English_Review"

In [ ]:
eng_review=chain_one.invoke(review)
eng_review

"I find the taste mediocre. The foam doesn't hold, it's strange. I buy the same ones in stores and the taste is much better...  \nOld batch or counterfeit!?"

In [ ]:
second_prompt = ChatPromptTemplate.from_template(
    "Can you summarize the following review in 1 sentence:"
    "\n\n{English_Review}"
)
# chain 2: input= English_Review and output= summary
chain_two = second_prompt|llm|StrOutputParser()  # output_key="summary"

In [ ]:
chain_two.invoke(eng_review)

'The reviewer finds the taste mediocre, the foam unappealing, and suspects the product might be old or counterfeit compared to better-tasting store-bought versions.'

In [ ]:
# prompt template 3: translate to english
third_prompt = ChatPromptTemplate.from_template(
    "What language is the following review:\n\n{Review}"
)
# chain 3: input= Review and output= language
chain_three = third_prompt|llm|StrOutputParser() # output_key="language"

In [ ]:
chain_three.invoke(review)

'The language of the review is French.'

In [ ]:
# prompt template 4: follow up message
fourth_prompt = ChatPromptTemplate.from_template(
    "Write a follow up response to the following "
    "summary in the specified language:"
    "\n\nSummary: {Summary}\n\nLanguage: {language}"
)
# chain 4: input= summary, language and output= followup_message
chain_four = fourth_prompt|llm|StrOutputParser() # output_key="followup_message"


In [ ]:
chain_four.invoke({"Summary":'The reviewer is disappointed with the taste and quality of the product, questioning if it is an old batch or counterfeit.',
                   "language":'French'})

"Je vous remercie pour votre retour concernant notre produit. Je suis désolé d'apprendre que son goût et sa qualité n'ont pas été à la hauteur de vos attentes. Nous prenons ces préoccupations très au sérieux, car la satisfaction de nos clients est notre priorité. Pourriez-vous nous fournir plus de détails sur votre expérience ? Cela nous aidera à comprendre ce qui s'est passé et à vérifier si le lot en question est conforme à nos standards. Nous sommes également prêts à offrir un remplacement ou un remboursement si cela peut vous aider. Merci de votre compréhension et nous espérons avoir de vos nouvelles bientôt."

In [ ]:
from langchain_core.runnables import RunnableParallel

runnable1 = RunnableParallel({
    "Summary": chain_one | chain_two,
    "language": chain_three,
   })

In [ ]:
runnable1.invoke(review)

{'Summary': 'The reviewer finds the taste mediocre and the foam ineffective, suspecting they may have received an old or counterfeit product compared to better-tasting versions purchased in stores.',
 'language': 'The review is written in French.'}

In [ ]:
final_chain= RunnableParallel({"Summary": chain_one | chain_two,"language": chain_three,}) |  {'followup_message':chain_four}


In [ ]:
#from langchain.callbacks.tracers import ConsoleCallbackHandler
final_chain.invoke(review) #,config={'callbacks': [ConsoleCallbackHandler()]})

{'followup_message': "Je vous remercie pour votre retour et vos commentaires sur notre produit. Nous sommes désolés d'apprendre que vous n'avez pas été satisfait du goût et de la mousse. Votre expérience est importante pour nous, et nous aimerions vous assurer que nous prenons vos préoccupations très au sérieux. Pourriez-vous nous fournir plus d'informations sur l'endroit où vous avez acheté le produit ? Cela nous aiderait à enquêter sur la qualité de ce lot en particulier. Nous souhaitons nous assurer que vous receviez toujours un produit de la meilleure qualité possible. Merci de votre compréhension et de votre soutien."}